# Examine Relationship Between Combine Results And DB Separation During Hitch/In/Out Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We have previously shown that some of our engineered features derived from NFL Combine player-tracking data have significant associations with a DBs ability to minimize separation on cutting routes.

We now want to test if that information tells us more than what general Combine results (40-yard dash time, 3 Cone drill time, etc.) tell us. That is, if coaches could learn just as much from simply looking at general Combine results, then they don't need our derived metrics. 

Therefore, we will test here if there are any significant relationships between general Combine metrics and a DBs ability to minimize space during cuts in hitch/in/out routes. We hypothesize that no Combine features will have any predictive power on a DBs ability to minimize space.

Predictors: General Combine Results

Outcome: Minimizing Separation At Cut During Hitch/In/Out Routes

**<span style="color: gold;">FINDINGS</span>**

No general Combine results had an association with DBs minimizing separation on go/post/corner routes.

**<span style="color: gold;">NOTES</span>**

Since we need to create outcome variables related to separation and since we need to know who is guarding who, we must use player-tracking data from the 2026 Big Data Bowl. That data only includes 2023 Regular season games. Therefore, our analysis will only include 2023 rookies during 2023 Regular season time.

While we usually only look at man coverage situations, we're too data starved to exclude zone coverage plays. Therefore, this analysis includes both man and zone coverage plays.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["HITCH", "IN", "OUT"]

<br><br/>
#### DB Separation Data From *11_BDB26_DB_Player_Tracking.ipynb*

In [ ]:
# =========================================
# DB Separation Data Among 2023 Rookies
# =========================================

db_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "11_2023_Rookie_DB_Player_Time_Level_Separation_Data.csv"))

# -----------------------------------------
# Identify Players Of Interest
# -----------------------------------------

players_of_interest = (
    db_separation_df
    .select("defender_nfl_id")
    .unique()
    .to_series()
    .to_list()
)

# ----------------------------------------
# Only Look At Specified Routes
# ----------------------------------------

db_separation_df = (
    db_separation_df
    .filter(pl.col("route").is_in(ROUTES_OF_INTEREST))
)

# ----------------------------------------
# Only Look At Man Coverage Situations
# ----------------------------------------

# db_separation_df = (
#     db_separation_df
#     .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
# )

print("Separation Data:")
print(f"    - Dimensions: {db_separation_df.shape}")
print(f"    - Number of Rookie DBs: {db_separation_df.select("defender_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {db_separation_df.select("player_play_id").n_unique()}")
db_separation_df

<br><br/>
#### Combine Features Data

In [ ]:
# =======================================
# Combine Results Data
# =======================================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)

combine_features_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "13_DB_Combine_Features_Data.csv"))
    .select("nfl_id", "best_w_decel")
)

combine_results_df = (
    combine_results_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "left"
    )
)

print("Combine Results Data:")
print(f"    - Dimensions: {combine_results_df.shape}")
print(f"    - Number of Players: {combine_results_df.select("nfl_id").n_unique()}")
combine_results_df

<br><br/>
## Data Cleaning
***

#### Get Data On Cuts And Separation At Cut From Targeted Hitch/In Routes

In [ ]:
# ==============================================================================
# Extract DB Closeness at the WR's Cut Apex for Hitch/In Routes
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Calculate WR Kinematics (Speed, Acceleration, and Downfield Velocity)
# ------------------------------------------------------------------------------
kinematics_df = (
    db_separation_df
    .sort(["player_play_id", "frame_id"])
    .with_columns(
        # Calculate raw directional velocities for the WR (dt = 0.1s at 10Hz)
        vx = (pl.col("wr_x") - pl.col("wr_x").shift(1).over("player_play_id")) / 0.1,
        vy = (pl.col("wr_y") - pl.col("wr_y").shift(1).over("player_play_id")) / 0.1,
    )
    .with_columns(
        # WR Speed is the magnitude of the velocity vector
        s = (pl.col("vx")**2 + pl.col("vy")**2).sqrt()
    )
    .with_columns(
        # Calculate true linear acceleration of the WR
        linear_accel = (pl.col("s") - pl.col("s").shift(1).over("player_play_id")) / 0.1
    )
    .with_columns(
        # Isolate true downfield velocity based on play direction
        v_downfield = pl.when(pl.col("play_direction") == "right")
                      .then(pl.col("vx"))
                      .otherwise(-pl.col("vx"))
    )
)

# ------------------------------------------------------------------------------
# 2. Isolate the WR's Cut Apex Frame Per Play
# ------------------------------------------------------------------------------
cut_apexes_df = (
    kinematics_df
    .filter(
        (pl.col("linear_accel").is_not_null()) & 
        (pl.col("v_downfield") > 4.0)
    )
    .group_by("player_play_id")
    .agg([
        # Find the frame where the WR slammed on the brakes the hardest
        pl.col("frame_id").filter(pl.col("linear_accel") == pl.col("linear_accel").min()).first().alias("cut_frame_id")
    ])
)

# ------------------------------------------------------------------------------
# 3. Extract DB Closeness and Calculate the Separation Delta
# ------------------------------------------------------------------------------
final_in_game_closeness_df = (
    kinematics_df
    .join(cut_apexes_df, on="player_play_id", how="inner")
    # Grab the metrics specifically at the cut apex or 0.5s right after it
    .filter(
        (pl.col("frame_id") == pl.col("cut_frame_id")) | 
        (pl.col("frame_id") == (pl.col("cut_frame_id") + 5))
    )
    .group_by("player_play_id")
    .agg([
        pl.col("defender_nfl_id").first(),
        pl.col("defender_name").first(),
        pl.col("defender_position").first(),
        # Distance to WR exactly at the moment of the cut plant
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == pl.col("cut_frame_id")).first().alias("distance_at_apex"),
        # Distance to WR 0.3 seconds after the cut
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("cut_frame_id") + 5)).first().alias("distance_post_cut")
    ])
    .with_columns(
        # Positive values = DB gave up separation. Negative values = DB closed the gap.
        separation_loss_delta = pl.col("distance_post_cut") - pl.col("distance_at_apex")
    )
)

# ------------------------------------------------------------------------------
# 4. Roll up to ONE row per DB to get their average coverage tightness
# ------------------------------------------------------------------------------
db_closeness_at_cut_df = (
    final_in_game_closeness_df
    .group_by("defender_nfl_id")
    .agg([
        pl.col("defender_name").first(),
        pl.col("defender_position").first(),
        pl.col("distance_at_apex").mean().alias("avg_distance_at_apex"),
        pl.col("distance_post_cut").mean().alias("avg_distance_post_cut"),
        pl.col("separation_loss_delta").mean().alias("avg_separation_loss_delta")
    ])
)

db_closeness_at_cut_df


<br><br/>
#### Add Combine Metrics To Player Separation Data

In [ ]:
analysis_df = (
    db_closeness_at_cut_df
    .join(
        combine_results_df, 
        left_on="defender_nfl_id", 
        right_on="nfl_id", 
        how = "inner"
    )
)

print("Final Analysis Dataset:")
print(f"    - Number of WRs: {analysis_df.select("defender_nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine Associations
***

We've previously shown that there is a statistically significant association between *best_w_decel* and *avg_separation_loss_delta*. We will now see if we can find a statistically significant relationship between *avg_separation_loss_delta* and our general Combine results.

<br><br/>
#### General Combine Results vs Separation Lost

In [ ]:
predictor = "ngs_athleticism_score" 
outcome = "avg_separation_loss_delta"

***Scatter Plot***

In [ ]:
fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols"
)

fig.show()

***Univariate Model***

In [ ]:
# ===============================================
# Univariate Regression
# ===============================================

# Does the relationship between best_w_decel and avg_separation_loss_delta
# hold if we adjust for general agility?

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

<br><br/>
## Deceleration vs NGS Athleticism
***

I want to check if our `best_w_decel` metric is strongly correlated with `ngs_athleticism_score`. I'm checking because I want to know if our new metric adds new and valuable information that coaches wouldn't have been able to know otherwise. 

<br><br/>
#### Examine Correlation

We see below that the correlation between `best_w_decel` and `ngs_athleticism_score` is low. Since they are not highly correlated, our new metric likely provides new, valuable information.

In [ ]:
# =============================================
# Correlation
# =============================================

print(f"Correlation Between best_w_decel And ngs_athleticism_score: {round(analysis_df.select(pl.corr("best_w_decel", "ngs_athleticism_score")).item(), 3)}")

<br><br/>
#### What Does Our Metric Add Beyond `ngs_atleticism_score`?

Does `best_w_decel` add useful information beyond what is already known by `ngs_athleticism_score`? We will check this by examining how much information is added to a model that includes `ngs_athleticism_score` when `best_w_decel` is added.

We see below that `best_w_decel` adds *very valuable* information. Additionally, when used with `best_w_decel`, `ngs_athleticism_score` becomes a helpful metric! R-squared shoots up when you add `best_w_decel` to `ngs_athleticism_score` and AIC decreases substantially when you add `best_w_decel` to `ngs_athleticism_score`.

In [ ]:
# ==========================================
# NGS Athleticism Alone
# ==========================================

predictor = "ngs_athleticism_score"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# ==========================================
# Deceleration Alone
# ==========================================

predictor = "best_w_decel"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# ===========================================
# NGS Athleticism Score And Deceleration
# ===========================================

predictors = [
    "ngs_athleticism_score",
    "best_w_decel", 
]

corr = round(analysis_df.select(pl.corr(predictors[0], predictors[1])).item(), 2)
vif = 1 / (1 - (corr**2))
print(f"Correlation Between {predictors[0]} and {predictors[1]}: {corr} (VIF = {round(vif, 2)})\n")

regression_df = analysis_df.filter(
    pl.all_horizontal(
        [pl.col(col).is_not_null() for col in predictors + [outcome]]
    )
)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictors]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())